# Lab 33: Custom Speech AI, Train Your Own Wake-Word Model

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
In Lab 19 you used a **pretrained** wake-word model (openWakeWord) that somebody else trained. It only knows the words it was trained on. In this lab you **train your own** keyword spotter, for your own words and your own voice. In Lab 17 Whisper wrote down everything you said. It is accurate but too slow and too big to **listen all the time**. A smart speaker listens all the time for **one word** ("Hey Siri", "OK Google") and only then wakes up the big model. This is **keyword spotting**. Here you you record the words *start* and *stop* (or your own words) and the sound of the room, turn the sound into a **picture of its frequencies** (a spectrogram), train a small neural network, and then let it listen live and switch a machine **on and off** by voice.

**By the end of this lab you can:**
1. Explain why a keyword spotter is small and always listening, while speech to text is big.
2. Cut recordings into one-second clips with an energy detector.
3. Turn a sound into a log-mel spectrogram, the "picture" a network can read.
4. Train a small neural network on your own recordings and read its accuracy and confusion matrix.
5. Run it live, tune the threshold, and measure missed words and false alarms.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Recording cells ask you to speak: do it when it says **SPEAK NOW**.

> **Privacy.** The microphone listens all the time while the live cell runs. Press **Stop** when you finish. Record only yourself and classmates who agree.

## 2. Background

**Why not use Whisper?** Whisper turns all speech into text. It needs a few seconds and a few hundred MB for every sentence. A keyword spotter answers one question only: *did I hear one of my words?* It can be tiny (this one has about 200 000 numbers), runs in a few milliseconds and can run all the time.

| | Whisper (Lab 17) | Keyword spotter (this lab) |
|---|---|---|
| Question | What was said? | Was it *start*, *stop* or something else? |
| Words | Any word | A few words you trained |
| Size and speed | Large, seconds | Tiny, milliseconds |
| Used for | Understanding a sentence | Waking up, simple voice commands |

**From sound to a picture.** A network cannot read 16 000 raw numbers per second well. The sound is first turned into a **spectrogram**: a picture where the horizontal axis is time, the vertical axis is frequency (low voice at the bottom, high at the top) and the brightness is the energy. Different words make different pictures. The steps are:

| Step | What happens |
|---|---|
| 1. Cut into short frames | 25 ms pieces, every 10 ms |
| 2. Fourier transform (`np.fft.rfft`) | Finds how much energy is in each frequency in each frame |
| 3. Mel filters | Groups the frequencies into 32 bands that follow the human ear (finer at low pitch) |
| 4. Logarithm | Our ear hears loudness on a log scale |

The result is a 32 x 49 picture for one second of sound (we join two frames to get 49 columns).

**Three classes.** The network chooses between your words and a class called **noise**. The noise class is as important as the words: it contains silence, room noise, clapping, typing and **other words**. Without it, the network must call every sound *start* or *stop*.

**Training data.** You record each word about 30 times. An **augmentation** step makes more examples from each clip by shifting it in time, changing the loudness and adding a little noise, so the network does not memorise one recording.

**Live listening.** The program listens all the time. When a sound starts after a quiet moment (an **onset**), it waits 0.8 seconds, takes the one second of sound around the onset, and asks the network. It acts only if the network is sure (the probability is above `THRESHOLD`). A **cooldown** stops it from acting twice on one word.

| Setting | If you raise it | If you lower it |
|---|---|---|
| `THRESHOLD` | Fewer false alarms, but more missed words | More words found, but more false alarms |
| `COOLDOWN` | Fewer repeats | Faster repeated commands |

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(19)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (20 min)

### Step 1: Settings and Microphone
Your microphone was set in Lab 17 (it is remembered). `WORDS` are the words to spot. Choose short words that **sound different**, not *stop* and *top*. `LIMIT_DB` is the loudness that counts as sound (see Lab 17, Part B).

In [ ]:
# ---------------- Settings you can change ----------------
WORDS = ["start", "stop"]    # the words to spot (2 or 3 short, different words)
CLIPS_PER_WORD = 30          # how many examples of each word you record
THRESHOLD = 0.80             # the network must be at least this sure (0 to 1)
COOLDOWN = 1.5               # seconds before the next word is accepted
LIMIT_DB = -38               # louder than this counts as sound (the same as in Lab 17)
# ----------------------------------------------------------
import csv
import glob
import os
import random
import threading
import time
from datetime import datetime

import numpy as np

import labview

RATE = 16000
CLASSES = WORDS + ["noise"]
print("Classes:", CLASSES, "| microphone:", labview.mic_get())

### Step 2: Check the Microphone
Record 2 seconds and read the level. Speak at the **second** half of the recording. The quiet part should be well below `LIMIT_DB` and your voice well above it.

In [ ]:
def level_db(x):
    """The loudness of a sound in decibels (0 is the maximum, a quiet room is about -60)."""
    return 20 * np.log10(max(float(np.sqrt(np.mean(np.square(x)))), 1e-6))


print("Stay quiet for one second, then say 'start' ...")
x = labview.record(2)
print(f"First second (quiet): {level_db(x[:RATE]):.0f} dB    second second (voice): {level_db(x[RATE:]):.0f} dB    limit: {LIMIT_DB} dB")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Level of the quiet room and of your voice:
- Is `LIMIT_DB` between the two? If not, what will you change it to?:


---
# Part B: Record Your Examples (45 min)

### Step 1: Cut a Recording into Clips
The function `cut_clips` finds every moment when a sound **starts after a quiet moment** (0.3 s) and cuts **one second** around it: 0.2 s before the start and 0.8 s after. This is exactly what the live program does later, so training and listening match. The cell checks it with a fake recording.

In [ ]:
def cut_clips(x, limit_db=None):
    """One-second clips (16 000 numbers each) that start 0.2 s before a sound begins after a quiet moment."""
    limit_db = LIMIT_DB if limit_db is None else limit_db
    frame = int(RATE * 0.02)
    loud = [level_db(x[i:i + frame]) > limit_db for i in range(0, len(x) - frame, frame)]
    clips, quiet = [], 99
    for index, flag in enumerate(loud):
        if flag and quiet >= 15:                             # 15 frames of 20 ms = 0.3 s of quiet before
            start = max(0, index * frame - int(0.2 * RATE))
            if start + RATE <= len(x):
                clips.append(x[start:start + RATE].copy())
        quiet = 0 if flag else quiet + 1
    return clips


# a fake recording: noise, then three short tones, one second apart
t = np.arange(int(0.4 * RATE)) / RATE
tone = (0.3 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
fake = np.random.randn(8 * RATE).astype(np.float32) * 0.001
for second in (1, 3, 5):
    fake[second * RATE:second * RATE + len(tone)] += tone
print("Clips found:", len(cut_clips(fake)), "(expected 3)")

### Step 2: Record the Words
`collect(word, seconds)` records for 45 seconds. In this time, say the word about **once every 2 seconds** with a pause, **30 times**. Vary the way you say it: loud and soft, fast and slow, near and far (20 to 60 cm), a little to the left and to the right. The clips are saved in `clips/<word>/`, so you do not need to record them again. Run it once per word.

In [ ]:
def save_clips(name, clips):
    os.makedirs(f"clips/{name}", exist_ok=True)
    first = len(glob.glob(f"clips/{name}/*.wav")) + 1
    for number, clip in enumerate(clips, first):
        labview.save_wav(clip, f"clips/{name}/{name}_{number:03d}.wav")
    print(f"Saved {len(clips)} clips of '{name}'. Now there are {len(glob.glob(f'clips/{name}/*.wav'))}.")


def collect(word, seconds=45):
    print(f"Say '{word}' about once every 2 seconds, {CLIPS_PER_WORD} times, with different loudness and distance.")
    time.sleep(2)
    print(">>> SPEAK NOW <<<")
    x = labview.record(seconds)
    print("Done.")
    clips = cut_clips(x)
    print("Clips found:", len(clips), f"(you wanted about {CLIPS_PER_WORD})")
    save_clips(word, clips)

In [ ]:
collect(WORDS[0])

In [ ]:
collect(WORDS[1])

If you have a third word, run `collect(WORDS[2])`. If a word has too few clips (less than 20), run it again. The new clips are added.

### Step 3: Record the Noise
The class **noise** teaches the network what is *not* a command. For 45 seconds make all kinds of other sounds: say **other words and sentences** (not your keywords), clap, tap the table, type, cough, and also stay quiet for a few seconds. The cell also adds random pieces of quiet.

In [ ]:
def collect_noise(seconds=45, extra=12):
    print("Make other sounds: talk about anything (NOT your keywords), clap, tap, type, be quiet at times.")
    time.sleep(2)
    print(">>> MAKE NOISE NOW <<<")
    x = labview.record(seconds)
    print("Done.")
    clips = cut_clips(x)
    for _ in range(extra):                                   # random pieces, many of them quiet
        start = random.randint(0, len(x) - RATE)
        clips.append(x[start:start + RATE].copy())
    save_clips("noise", clips)


collect_noise()

Look at the number of clips of each class and **listen** to one clip of each. A wrong clip (a word in the wrong class) will confuse the network, so check them.

In [ ]:
for name in CLASSES:
    files = sorted(glob.glob(f"clips/{name}/*.wav"))
    print(f"{name:<8} {len(files)} clips")
    if files:
        print("  one example:", files[len(files) // 2])
        labview.play(files[len(files) // 2], speaker=False)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Number of clips of each class:
- Was every clip the right word? How did you check?:
- How did you vary the way you said the words?:


---
# Part C: Train the Network (45 min)

### Step 1: The Spectrogram
`features` turns a clip into the 32 x 49 picture. `mel_filters` makes the 32 bands. Run the cell and look at the pictures of one clip of each class: the words have different shapes.

In [ ]:
import cv2
from IPython.display import Image, display


def mel_filters(n_mels=32, n_fft=400, rate=RATE):
    """32 triangle filters that group the frequencies like the human ear."""
    to_mel = lambda f: 2595 * np.log10(1 + f / 700)
    to_hz = lambda m: 700 * (10 ** (m / 2595) - 1)
    points = to_hz(np.linspace(to_mel(60), to_mel(rate / 2), n_mels + 2))
    bins = np.floor((n_fft + 1) * points / rate).astype(int)
    bank = np.zeros((n_mels, n_fft // 2 + 1), np.float32)
    for m in range(1, n_mels + 1):
        left, centre, right = bins[m - 1], bins[m], bins[m + 1]
        for k in range(left, centre):
            bank[m - 1, k] = (k - left) / max(1, centre - left)
        for k in range(centre, right):
            bank[m - 1, k] = (right - k) / max(1, right - centre)
    return bank


BANK = mel_filters()
WINDOW = np.hanning(400).astype(np.float32)


def features(clip):
    """A one-second clip -> a 32 x 49 picture (log-mel spectrogram, normalised)."""
    clip = np.pad(clip, (0, max(0, RATE - len(clip))))[:RATE]
    frames = np.lib.stride_tricks.sliding_window_view(clip, 400)[::160] * WINDOW        # 25 ms frames every 10 ms
    power = np.abs(np.fft.rfft(frames, axis=1)) ** 2                                    # energy of each frequency
    mel = np.log10(power @ BANK.T + 1e-6).T                                              # 32 bands, log scale
    mel = mel[:, :98].reshape(32, 49, 2).mean(axis=2)                                    # join two frames: 49 columns
    return ((mel - mel.mean()) / (mel.std() + 1e-5)).astype(np.float32)


def show_features(name):
    files = sorted(glob.glob(f"clips/{name}/*.wav"))
    clip = labview.load_wav(files[len(files) // 2])[0]
    f = features(clip)
    picture = cv2.resize(((f - f.min()) / (f.max() - f.min() + 1e-5) * 255).astype(np.uint8)[::-1], (392, 256), interpolation=cv2.INTER_NEAREST)
    cv2.putText(picture, name, (6, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.7, 255, 2)
    return picture


pictures = [show_features(name) for name in CLASSES if glob.glob(f"clips/{name}/*.wav")]
display(Image(data=cv2.imencode(".png", cv2.applyColorMap(np.hstack(pictures), cv2.COLORMAP_VIRIDIS))[1].tobytes()))
print("Shape of the picture of one clip:", features(np.zeros(RATE, np.float32)).shape)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Do the pictures of the words look different from each other and from noise?:
- Why do we take the logarithm of the energy?:


### Step 2: Make the Training Data
The cell loads all clips, keeps 20 % of each class for **checking** (the network never trains on them), and makes 8 augmented copies of each training clip: shifted by up to 0.1 s, louder or softer, with a little noise.

In [ ]:
def augment(clip):
    shift = int(random.uniform(-0.1, 0.1) * RATE)
    clip = np.roll(clip, shift)
    if shift > 0:
        clip[:shift] = 0
    elif shift < 0:
        clip[shift:] = 0
    return clip * random.uniform(0.5, 1.5) + np.random.randn(RATE).astype(np.float32) * random.uniform(0, 0.004)


def load_data(val_share=0.2, copies=8, seed=1):
    rng = random.Random(seed)
    train_x, train_y, val_x, val_y = [], [], [], []
    for number, name in enumerate(CLASSES):
        files = sorted(glob.glob(f"clips/{name}/*.wav"))
        rng.shuffle(files)
        n_val = max(1, round(len(files) * val_share)) if len(files) > 1 else 0
        for position, path in enumerate(files):
            clip = labview.load_wav(path)[0]
            if position < n_val:
                val_x.append(features(clip)); val_y.append(number)
            else:
                train_x.append(features(clip)); train_y.append(number)
                for _ in range(copies):
                    train_x.append(features(augment(clip.copy()))); train_y.append(number)
    return (np.array(train_x), np.array(train_y), np.array(val_x), np.array(val_y))


train_x, train_y, val_x, val_y = load_data()
print("Training pictures:", len(train_x), "(with augmentation) | checking pictures:", len(val_x))
for number, name in enumerate(CLASSES):
    print(f"  {name:<8} train {int((train_y == number).sum()):>4}   check {int((val_y == number).sum()):>3}")

### Step 3: The Network and the Training
The network is small: it flattens the 32 x 49 picture (1568 numbers), has one hidden layer of 128 neurons, and has one output for each class. `softmax` turns the outputs into probabilities that add up to 1. The training shows the **loss** (how wrong the network is, lower is better) and the **accuracy** on the checking clips. It takes less than a minute.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
net = nn.Sequential(nn.Flatten(), nn.Linear(32 * 49, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, len(CLASSES)))
print("Numbers the network learns:", sum(p.numel() for p in net.parameters()))

optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
loss_function = nn.CrossEntropyLoss()
tx, ty = torch.tensor(train_x), torch.tensor(train_y)
vx, vy = torch.tensor(val_x), torch.tensor(val_y)

for epoch in range(1, 61):
    net.train()
    order = torch.randperm(len(tx))
    total = 0.0
    for i in range(0, len(tx), 32):
        batch = order[i:i + 32]
        optimizer.zero_grad()
        loss = loss_function(net(tx[batch]), ty[batch])
        loss.backward()
        optimizer.step()
        total += loss.item() * len(batch)
    if epoch % 10 == 0 or epoch == 1:
        net.eval()
        with torch.no_grad():
            accuracy = (net(vx).argmax(1) == vy).float().mean().item()
        print(f"epoch {epoch:>2}   loss {total / len(tx):.3f}   accuracy on the checking clips {100 * accuracy:.0f} %")

torch.save({"net": net.state_dict(), "classes": CLASSES}, "kws.pt")
print("Saved as kws.pt")

### Step 4: Where Does It Go Wrong?
The **confusion matrix** counts, for every real class (rows), what the network said (columns). A perfect network has numbers only on the diagonal.

In [ ]:
def predict(clip):
    """Probabilities of each class for a one-second clip."""
    net.eval()
    with torch.no_grad():
        return torch.softmax(net(torch.tensor(features(clip))[None]), dim=1)[0].numpy()


net.eval()
with torch.no_grad():
    guessed = net(vx).argmax(1).numpy()
print(f"{'real \\ said':<12}" + "".join(f"{name:>8}" for name in CLASSES))
for number, name in enumerate(CLASSES):
    row = [int(((val_y == number) & (guessed == k)).sum()) for k in range(len(CLASSES))]
    print(f"{name:<12}" + "".join(f"{v:>8}" for v in row))
print(f"Accuracy on the checking clips: {100 * (guessed == val_y).mean():.0f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Accuracy on the checking clips:
- Which class was mixed up most (look at the matrix)?:
- Loss at the first and the last epoch:
- Why do we check with clips that the network never trained on?:


---
# Part D: Listen Live and Control a Machine (40 min)

### Step 1: The Live Listener
The program listens all the time. It waits for a sound onset, waits 0.8 s, cuts the clip, asks the network and, if it is sure, acts:
- **start** switches the machine to **RUNNING**, **stop** switches it to **STOPPED**. If you use other words, the first word means on and the others mean off.
- Every accepted word is saved in `commands.csv` with the time and the probability.

`Listener` runs in a background thread. Run the cell and speak. The status line shows what it heard. Press **Stop** when you are done.

In [ ]:
import ipywidgets as widgets


def classify_stream(stream, state):
    """One step of the listener: returns (word, probability) when a word is found, otherwise None. state keeps the memory."""
    now = time.time()
    loud = level_db(stream.latest(0.05)) > LIMIT_DB
    result = None
    if loud:
        if state["onset"] is None and now - state["last_loud"] >= 0.3:
            state["onset"] = now                                  # a sound starts after a quiet moment
        state["last_loud"] = now
    if state["onset"] is not None and now - state["onset"] >= 0.8:
        state["onset"] = None
        probabilities = predict(stream.latest(1.0))
        best = int(probabilities.argmax())
        if CLASSES[best] != "noise" and probabilities[best] >= THRESHOLD and now - state["last_word"] >= COOLDOWN:
            state["last_word"] = now
            result = (CLASSES[best], float(probabilities[best]))
        state["last_seen"] = (CLASSES[best], float(probabilities[best]))
    return result


def log_command(word, probability):
    first = not os.path.exists("commands.csv")
    with open("commands.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "word", "probability"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), word, round(probability, 2)])


class Listener:
    def __init__(self):
        self.stream = labview.MicStream(RATE)
        self.state = {"onset": None, "last_loud": 0.0, "last_word": 0.0, "last_seen": None}
        self.machine = "STOPPED"
        self.counts = {"words": 0}
        self.stop_flag = threading.Event()
        self.screen = widgets.Label(value="Listening ...")
        self.button = widgets.Button(description="Stop", button_style="danger")
        self.button.on_click(lambda _: self.stop())
        from IPython.display import display
        display(widgets.VBox([self.button, self.screen]))
        threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        while not self.stop_flag.is_set():
            time.sleep(0.05)
            if not self.stream.alive():
                self.screen.value = "The microphone stopped: " + self.stream.error()
                break
            found = classify_stream(self.stream, self.state)
            if found:
                word, probability = found
                self.counts["words"] += 1
                self.machine = "RUNNING" if word == WORDS[0] else "STOPPED"
                log_command(word, probability)
            seen = self.state["last_seen"]
            self.screen.value = (f"MACHINE: {self.machine}    accepted words: {self.counts['words']}    "
                                 + (f"last sound: {seen[0]} ({100 * seen[1]:.0f} %)" if seen else "last sound: ---"))

    def stop(self):
        self.stop_flag.set()
        self.stream.stop()
        self.screen.value += "   [stopped]"


listener = Listener()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Does the machine switch when you say the words? How many tries out of 10 for each word?:
- What does the status line show when you say another word or clap?:
- What delay do you feel between saying a word and the switch? Why is there a delay?:


### Step 2: Tune the Threshold
Change `THRESHOLD` and run the settings cell and the cell of Step 1 again (stop the old listener first). Try 0.6, 0.8 and 0.95. At each value say each word 10 times and talk or make noise for 1 minute.

| THRESHOLD | Words found out of 10 (*start*) | Words found out of 10 (*stop*) | False alarms in 1 minute |
|---|---|---|---|
| 0.60 | | | |
| 0.80 | | | |
| 0.95 | | | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Results of the table:
- Which THRESHOLD is best for your use and why?:
- What is worse for a machine: a missed word or a false alarm? Does it depend on the machine?:


After you pressed Stop, see the log:

In [ ]:
if os.path.exists("commands.csv"):
    with open("commands.csv") as f:
        print(f.read())
else:
    print("No commands yet.")

---
# Part E: Make It Your Own, Test and Present (30 min)

### Step 1: Make It Your Own
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the live cell again.

- Add a **third word** (for example *go* or *left*), record it, train again, and give it an action.
- Make the network **better**: record 30 more clips with different distances, train again and compare the accuracy.
- Show a **picture of a lamp** (use `IPython.display` or a coloured label) that is green when RUNNING and red when STOPPED.
- Send the command to your **phone** with `labview.mqtt()` and `client.publish("machine/state", "RUNNING")` (see Lab 16).
- Use **Whisper after the wake word**: when the network hears *start*, record 4 seconds and turn them into text with Lab 17. This is how a smart speaker works. (Lab 19 does it.)
- Let a classmate say the words. Does it work for them? Add their clips to the training data.

> **Tip.** Change one thing at a time. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


### Step 2: Test
Stop the listener and delete the old log. Then test every situation with the listener running and write what happened.

In [ ]:
if os.path.exists("commands.csv"):
    os.remove("commands.csv")
print("Clean.")

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | Say each word normally, 30 cm, 10 times | Found 9 or 10 times | |
| 2 | Say each word softly | Found? | |
| 3 | Say each word from 1.5 metres | Found? | |
| 4 | A classmate says each word | Found? | |
| 5 | Say a word that sounds similar (for example *top* for *stop*) | Nothing | |
| 6 | Talk normally for 1 minute (no keywords) | Nothing (zero false alarms) | |
| 7 | Music or a noisy room for 1 minute | Nothing | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 3:
- Test 4: how did a different voice work?:
- Test 5 to 7: any false alarms? When?:
- What would you do to make it work for many different people?:


### Step 3: Report and Present
Fill in the report. Then give a **3-minute demonstration**: show the spectrograms, the training numbers and the confusion matrix, then control the machine with your voice and show the test table.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Words and the number of clips of each class:
- Accuracy on the checking clips and the confusion matrix result:
- THRESHOLD and COOLDOWN used:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** The Jetson can now hear. In the next labs you give the Jetson a **language model** (Labs 20 to 26) and join everything into assistants (Lab 22).

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The fake recording gives 3 clips and your recordings give about 30 clips per word | |
| 2 | The noise class is recorded with other words and sounds | |
| 3 | The spectrogram pictures are shown and explained | |
| 4 | The network is trained and the loss goes down | |
| 5 | The accuracy and the confusion matrix are read | |
| 6 | The live listener switches the machine with your words | |
| 7 | The threshold table is complete and a threshold is chosen with a reason | |
| 8 | At least one improvement works and you explain the code you changed | |
| 9 | The test table is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. Why is a keyword spotter used to wake up a speech-to-text model, and not the other way round?:
- 2. What is a spectrogram? What do the two axes and the brightness mean?:
- 3. Why does the network need a `noise` class? What would happen without it?:
- 4. What is augmentation and why do we use it?:
- 5. Explain what the confusion matrix shows. Where do you look for the number of mistakes?:
- 6. Why does the listener wait 0.8 seconds after the onset before it asks the network?:
- 7. You raise THRESHOLD from 0.8 to 0.95. What happens to missed words and false alarms?:
- 8. The spotter works for you but not for a classmate. Why, and what do you do?:
- 9. Which is a good use of keyword spotting? (a) Turning a lamp on with the word 'light' (b) Reading a book aloud (c) Writing a meeting report (d) Translating a movie:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `Cannot record ...` | Wrong microphone | Set it again as in Lab 17 (`labview.audio_devices()`, then `labview.mic_set(...)`) |
| Too few clips found (less than 20) | `LIMIT_DB` is too high, or you spoke too softly or too fast | Lower `LIMIT_DB` a little (for example -42). Wait about 2 seconds between words. |
| Too many clips (more than you said) | `LIMIT_DB` is too low, so noise counts | Raise `LIMIT_DB`. Record in a quieter place. |
| `IndexError` or an empty class | A class has no clips | Run `collect(...)` for every word and `collect_noise()` |
| The accuracy is below 80 % | Too few or unclear clips, words that sound alike | Record more clips, check them by listening, use words that sound different |
| The accuracy is 100 % but live it fails | Too few checking clips, or the live sound is different | Record clips at the distance you will use. Add more noise. |
| The machine never switches | THRESHOLD too high or `LIMIT_DB` not matching | Lower THRESHOLD to 0.6. Read the status line: it shows what it heard. |
| The machine switches by itself | THRESHOLD too low, or noise clips are missing | Raise THRESHOLD. Record more noise clips with other words. |
| `The microphone stopped` | Another notebook uses the microphone | Stop other recording cells. Shut down other kernels. |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(19)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Speech Commands dataset (Google): https://www.tensorflow.org/datasets/catalog/speech_commands
- Hello Edge, keyword spotting on small devices (paper): https://arxiv.org/abs/1711.07128
- Mel scale: https://en.wikipedia.org/wiki/Mel_scale
- PyTorch tutorials: https://pytorch.org/tutorials/